# ME5311 Project 1 — Spatiotemporal Vector Field Data Analysis

**Author**: Yu Huize · **Course**: NUS ME5311

---

This notebook performs a complete data-driven analysis of 2D spatiotemporal vector field datasets, including:

| Step | Content | Project Questions |
|------|---------|-------------------|
| Step 0 | Data loading, mean field/fluctuation separation, vorticity/divergence | Preprocessing |
| Step 1 | SVD analysis: singular value spectrum, spatial modes, temporal coefficient spectrum | Q1 Leading spatial structure |
| Step 2 | Fourier spectral analysis: 2D PSD, radial spectrum, temporal PSD | Q2 Energy distribution & Q3 Periodicity |
| Step 3 | Symmetry & anisotropy diagnosis | Q4 Symmetry |

In [ ]:
%matplotlib inline

import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Add project root directory to sys.path to import src modules
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

# Figure saving directory
FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

# Unified plotting style
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'image.cmap': 'RdBu_r',
})

from src import data_loader as dl
print("✅ Environment setup complete")

---
## Step 0: Data Loading and Preprocessing

Load raw data `vector_64.npy` and compute:
- **Time-averaged field** $\bar{\mathbf{u}}(x,y)$
- **Fluctuation field** $\mathbf{u}' = \mathbf{u} - \bar{\mathbf{u}}$
- **Vorticity** $\omega = \partial_x u_y - \partial_y u_x$
- **Divergence** $\nabla \cdot \mathbf{u} = \partial_x u_x + \partial_y u_y$

In [ ]:
# Load and preprocess
bundle = dl.load_and_preprocess()
raw         = bundle['raw']           # (15000, 64, 64, 2)
mean_field  = bundle['mean_field']    # (64, 64, 2)
fluctuation = bundle['fluctuation']   # (15000, 64, 64, 2)
data_matrix = bundle['data_matrix']   # (8192, 15000)
vorticity   = bundle['vorticity']     # (15000, 64, 64)
divergence  = bundle['divergence']    # (15000, 64, 64)

# Basic statistics
print(f"Data shape: {raw.shape}")
print(f"Data type: {raw.dtype}")
print(f"Value range: [{raw.min():.4f}, {raw.max():.4f}]")
print(f"ux mean: {raw[...,0].mean():.6f}, std: {raw[...,0].std():.4f}")
print(f"uy mean: {raw[...,1].mean():.6f}, std: {raw[...,1].std():.4f}")

### 0.1 Time-Averaged Field (Mean Field)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ux component
ux_mean = mean_field[..., 0]
im0 = axes[0].imshow(ux_mean, origin='lower', cmap='RdBu_r',
                      vmin=-np.abs(ux_mean).max(), vmax=np.abs(ux_mean).max())
axes[0].set_title('Mean field $\\bar{u}_x$')
fig.colorbar(im0, ax=axes[0], fraction=0.046)

# uy component
uy_mean = mean_field[..., 1]
im1 = axes[1].imshow(uy_mean, origin='lower', cmap='RdBu_r',
                      vmin=-np.abs(uy_mean).max(), vmax=np.abs(uy_mean).max())
axes[1].set_title('Mean field $\\bar{u}_y$')
fig.colorbar(im1, ax=axes[1], fraction=0.046)

# Velocity magnitude + quiver
mag = np.sqrt(ux_mean**2 + uy_mean**2)
im2 = axes[2].imshow(mag, origin='lower', cmap='viridis')
step = 3
Y, X = np.mgrid[0:64, 0:64]
axes[2].quiver(X[::step, ::step], Y[::step, ::step],
               ux_mean[::step, ::step], uy_mean[::step, ::step],
               color='k', alpha=0.7)
axes[2].set_title('Mean field magnitude + quiver')
fig.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_aspect('equal')
fig.tight_layout()
fig.savefig(FIG_DIR / 'step0_mean_field.png')
plt.show()

#### 📊 How to Interpret This Figure

| Item | Explanation |
|------|-------------|
| **Figure meaning** | Time-averaged vector field over 15000 time steps. Left: $u_x$ component, middle: $u_y$ component, right: velocity magnitude + arrow field |
| **How to read** | Red/blue indicates positive/negative values; arrow direction and length represent local flow direction and speed magnitude |
| **Good signal** | If mean field exhibits **obvious spatial periodicity**, external forcing leaves clear imprints on average |
| **Bad signal** | If mean field is near zero (all blue), mean state is symmetric, forcing manifests mainly in fluctuations |
| **Key conclusion** | Check for repeated streaks/vortex structures; estimate spatial period (grid points per repetition) |

### 0.2 Snapshots at Different Times

In [ ]:
time_indices = [0, 3000, 7500, 12000]  # Different time steps
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

for col, t_idx in enumerate(time_indices):
    t_val = t_idx * dl.DT
    snap = raw[t_idx]
    mag = np.sqrt(snap[...,0]**2 + snap[...,1]**2)

    # Top row: velocity magnitude
    im = axes[0, col].imshow(mag, origin='lower', cmap='viridis')
    axes[0, col].set_title(f'|u| at t={t_val:.0f}')
    fig.colorbar(im, ax=axes[0, col], fraction=0.046)

    # Bottom row: vorticity
    im = axes[1, col].imshow(vorticity[t_idx], origin='lower', cmap='RdBu_r',
                              vmin=-np.abs(vorticity[t_idx]).max(),
                              vmax=np.abs(vorticity[t_idx]).max())
    axes[1, col].set_title(f'$\\omega$ at t={t_val:.0f}')
    fig.colorbar(im, ax=axes[1, col], fraction=0.046)

for ax in axes.flat:
    ax.set_aspect('equal')
fig.suptitle('Snapshots at different times (top: magnitude, bottom: vorticity)', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / 'step0_snapshots.png')
plt.show()

#### 📊 How to Interpret This Figure

| Item | Explanation |
|------|-------------|
| **Figure meaning** | 4 snapshots of velocity magnitude (top row) and vorticity field (bottom row) at different times, showing temporal evolution |
| **How to read** | Top: brighter colors = higher speeds; Bottom: red/blue = positive/negative vorticity (counterclockwise/clockwise rotation) |
| **Good signal** | Patterns change over time but maintain similar spatial scales → system has a **characteristic length scale** |
| **Key conclusion** | Check for obvious spatial periodicity and similarity between different time instants |

### 0.3 Divergence Check (Is Flow Approximately Incompressible?)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Divergence snapshot
im = axes[0].imshow(divergence[0], origin='lower', cmap='RdBu_r',
                     vmin=-np.abs(divergence[0]).max(),
                     vmax=np.abs(divergence[0]).max())
axes[0].set_title('Divergence $\\nabla \\cdot \\mathbf{u}$ at t=0')
axes[0].set_aspect('equal')
fig.colorbar(im, ax=axes[0], fraction=0.046)

# Divergence statistics
div_rms = np.sqrt(np.mean(divergence**2, axis=(1, 2)))
axes[1].plot(np.arange(dl.NT) * dl.DT, div_rms, lw=0.5)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('RMS divergence')
axes[1].set_title('RMS divergence over time')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'step0_divergence.png')
plt.show()
print(f"Global divergence RMS: {np.sqrt(np.mean(divergence**2)):.6f}")
print(f"Global velocity RMS: {np.sqrt(np.mean(raw**2)):.6f}")
print(f"Divergence/velocity ratio: {np.sqrt(np.mean(divergence**2)) / np.sqrt(np.mean(raw**2)):.6f}")

#### 📊 How to Interpret This Figure

| Item | Explanation |
|------|-------------|
| **Figure meaning** | Left: divergence field at t=0; Right: RMS divergence over time |
| **How to read** | $\nabla \cdot \mathbf{u}$ closer to zero → flow closer to **incompressible** (like Navier-Stokes) |
| **Good signal** | Divergence/velocity ratio ≪ 1 (e.g., < 0.01) → approximately incompressible |
| **Bad signal** | Ratio ≈ 1 → flow has obvious compressibility, analysis must consider this |
| **Key conclusion** | Determine if system is incompressible flow, which affects subsequent physical interpretation |

---
## Step 1: SVD Analysis — Leading Spatial Structure (Project Question 1)

Perform economic SVD decomposition on data matrix $A \in \mathbb{R}^{8192 \times 15000}$ from fluctuation field:

$$A = U \Sigma V^T$$

- $U$: Spatial modes (left singular vectors), revealing leading spatial structure
- $\Sigma$: Singular values, reflecting "energy" of each mode
- $V^T$: Temporal coefficients (right singular vectors), reflecting temporal evolution of each mode

In [ ]:
# Economic SVD (compute only min(N,T)=8192 singular values)
print("Performing SVD decomposition (may take 1-2 minutes)…")
U, sigma, Vt = np.linalg.svd(data_matrix, full_matrices=False)
print(f"✅ SVD complete: U={U.shape}, sigma={sigma.shape}, Vt={Vt.shape}")

# Energy statistics
energy = sigma**2
cum_energy = np.cumsum(energy) / energy.sum()
n95 = int(np.searchsorted(cum_energy, 0.95)) + 1
n99 = int(np.searchsorted(cum_energy, 0.99)) + 1
print(f"Modes needed to reach 95% energy: {n95}")
print(f"Modes needed to reach 99% energy: {n99}")

### 1.1 Singular Value Spectrum and Cumulative Energy

In [ ]:
n_show = min(100, len(sigma))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: singular values (log scale)
axes[0].semilogy(np.arange(1, n_show+1), sigma[:n_show], 'o-', ms=3, color='steelblue')
axes[0].set_xlabel('Mode index $k$')
axes[0].set_ylabel('Singular value $\\sigma_k$')
axes[0].set_title('Singular value spectrum (log scale)')
axes[0].grid(True, alpha=0.3)

# Right plot: cumulative energy fraction
axes[1].plot(np.arange(1, n_show+1), cum_energy[:n_show]*100, 's-', ms=3, color='darkorange')
axes[1].axhline(95, color='r', ls='--', lw=1, label=f'95% → {n95} modes')
axes[1].axhline(99, color='green', ls='--', lw=1, label=f'99% → {n99} modes')
axes[1].set_xlabel('Number of modes $K$')
axes[1].set_ylabel('Cumulative energy (%)')
axes[1].set_title('Cumulative energy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'step1_singular_values.png')
plt.show()

# Print energy fractions of first 10 modes
print("\nEnergy fractions of first 10 modes:")
for i in range(10):
    print(f"  Mode {i+1}: σ={sigma[i]:.2f}, fraction={energy[i]/energy.sum()*100:.2f}%, cumulative={cum_energy[i]*100:.2f}%")

#### 📊 How to Interpret Singular Value Spectrum

| Item | Explanation |
|------|-------------|
| **Figure meaning** | Left: magnitude of first 100 singular values (log scale); Right: cumulative energy growth with mode number |
| **How to read** | Faster singular value decay → "lower rank" data, few modes capture most information |
| **Good performance** | First few singular values much larger than rest (steep curve drop) → **few clear dominant structures** |
| **Poor performance** | Slow singular value decay (flat curve) → complex/chaotic structure, hard to approximate with few modes |
| **Key conclusion** | ① How many modes to reach 95%/99% energy? ② Is there an obvious "knee" (energy gap)? |
| **Judgment** | <20 modes for 95% → highly structured system; >100 modes → more complex system |

### 1.2 Leading Spatial Modes

In [ ]:
n_modes = 6
half = dl.NY * dl.NX  # 4096

fig, axes = plt.subplots(n_modes, 2, figsize=(12, 3.5*n_modes))
for i in range(n_modes):
    mode_vec = U[:, i]
    ux_mode = mode_vec[:half].reshape(dl.NY, dl.NX)
    uy_mode = mode_vec[half:].reshape(dl.NY, dl.NX)

    for j, (comp, label) in enumerate([(ux_mode, '$u_x$'), (uy_mode, '$u_y$')]):
        ax = axes[i, j]
        vmax = np.abs(comp).max()
        im = ax.imshow(comp, origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        pct = energy[i] / energy.sum() * 100
        ax.set_title(f'Mode {i+1} ({pct:.1f}% energy) — {label}')
        ax.set_aspect('equal')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(FIG_DIR / 'step1_spatial_modes.png')
plt.show()

#### 📊 How to Interpret Spatial Modes

| Item | Explanation |
|------|-------------|
| **Figure meaning** | Spatial distribution of first 6 SVD modes; left column: $u_x$ component, right column: $u_y$ component |
| **How to read** | Each mode is a "spatial basis function"; red/blue alternation shows positive/negative regions; earlier modes are more important |
| **Good signal** | Modes exhibit **clear spatial periodicity** (regular stripes/checkerboard/vortex arrays) → system has a characteristic scale |
| **Bad signal** | Modes look like random noise → low energy fraction, not a dominant structure |
| **Key conclusion** | ① Count spatial oscillations (peak-trough alternations) → characteristic wavenumber |  
| | ② Observe if $u_x$ and $u_y$ modes have 90° rotation relationship (rotate symmetry) |
| | ③ Check if modes appear in pairs (e.g., Mode 1&2 similar but phase-shifted) → traveling waves |

### 1.3 Temporal Coefficients

In [ ]:
fig, axes = plt.subplots(n_modes, 1, figsize=(14, 2.5*n_modes), sharex=True)
t_arr = np.arange(dl.NT) * dl.DT

for i in range(n_modes):
    coeff = sigma[i] * Vt[i, :]
    ax = axes[i]
    ax.plot(t_arr, coeff, lw=0.4, color='steelblue')
    ax.set_ylabel(f'Mode {i+1}')
    ax.grid(True, alpha=0.3)
    # Show zoomed view
    ax_inset = ax.inset_axes([0.75, 0.1, 0.23, 0.8])
    t_zoom = slice(0, 500)  # First 100 time units
    ax_inset.plot(t_arr[t_zoom], coeff[t_zoom], lw=0.6, color='coral')
    ax_inset.set_xlim(0, 100)
    ax_inset.tick_params(labelsize=7)

axes[-1].set_xlabel('Time')
axes[0].set_title('SVD temporal coefficients $\\sigma_k v_k(t)$ (inset: zoom t=0~100)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'step1_temporal_coeff.png')
plt.show()

#### 📊 How to Interpret Temporal Coefficients

| Item | Explanation |
|------|-------------|
| **Figure meaning** | Temporal evolution of first 6 modes' coefficients $a_k(t) = \sigma_k v_k(t)$; right panel: zoom of first 100 time units |
| **How to read** | Each curve is the time-varying amplitude of that spatial mode; more regular oscillation → clearer temporal frequency |
| **Good signal** | **Regular oscillations** → quasi-periodic motion; **stable amplitude** → statistically steady state |
| **Bad signal** | Amplitude drifts or jumps over time → transient processes or non-stationary behavior |
| **Key conclusion** | ① Do all modes oscillate at same frequency? ② Is there low-frequency modulation? ③ Is system statistically steady? |

### 1.4 Spectral Analysis of Temporal Coefficients

In [ ]:
fig, axes = plt.subplots(n_modes, 1, figsize=(14, 2.5*n_modes), sharex=True)
freqs_mode = np.fft.rfftfreq(dl.NT, d=dl.DT)

for i in range(n_modes):
    coeff = sigma[i] * Vt[i, :]
    fhat = np.fft.rfft(coeff)
    psd = np.abs(fhat)**2 / dl.NT
    ax = axes[i]
    ax.semilogy(freqs_mode[1:], psd[1:], lw=0.6, color='steelblue')  # Skip DC
    # Mark peak frequency
    peak_idx = np.argmax(psd[1:]) + 1
    ax.axvline(freqs_mode[peak_idx], color='red', ls='--', lw=1,
               label=f'peak f={freqs_mode[peak_idx]:.4f}')
    ax.set_ylabel(f'Mode {i+1}')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Frequency $f$')
axes[0].set_title('PSD of SVD temporal coefficients')
fig.tight_layout()
fig.savefig(FIG_DIR / 'step1_mode_temporal_psd.png')
plt.show()

#### 📊 How to Interpret Temporal Spectrum of Modes

| Item | Explanation |
|------|-------------|
| **Figure meaning** | Power spectral density of first 6 SVD mode temporal coefficients; x-axis: frequency, y-axis: power (log scale) |
| **How to read** | Sharp peaks (spikes) = mode oscillates **regularly** at that frequency; broadband spectrum = **chaotic/turbulent** |
| **Good signal** | Clear discrete frequency peaks → system has **definite time scales** (corresponding period $T = 1/f$) |
| **Bad signal** | No obvious peaks, power spreads continuously → temporal dynamics more complex/chaotic |
| **Key conclusion** | ① Do all modes have the same dominant frequency? ② If Modes 1&2 share frequency with possible phase difference → traveling waves ③ What time period corresponds to peak frequency? |

---
## Step 2: Fourier Spectral Analysis — Energy Distribution and Periodicity Inference (Project Questions 2 & 3)

Perform Fourier transforms on fluctuation field in **spatial domain** (2D FFT) and **temporal domain** (temporal FFT), analyzing:
- Energy distribution across spatial scales → **Radial power spectrum** $\text{PSD}(k)$
- Presence of external periodic forcing → **Prominent spectral peaks**
- Temporal scale characteristics → **Temporal frequency PSD**

### 2.1 2D Spatial Power Spectrum

In [ ]:
# Compute time-averaged 2D PSD for ux, uy fluctuation fields
def spatial_psd_2d(data_4d, component):
    field = data_4d[..., component]
    fhat = np.fft.fft2(field, axes=(1, 2))
    psd = np.mean(np.abs(fhat)**2, axis=0) / (dl.NX * dl.NY)
    return psd

psd_ux_2d = spatial_psd_2d(fluctuation, 0)
psd_uy_2d = spatial_psd_2d(fluctuation, 1)
psd_total_2d = psd_ux_2d + psd_uy_2d

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, psd, title in zip(axes, [psd_ux_2d, psd_uy_2d, psd_total_2d],
                           ['$u_x$ PSD', '$u_y$ PSD', 'Total PSD']):
    display = np.log10(np.fft.fftshift(psd) + 1e-30)
    n = psd.shape[0]
    extent = [-n//2, n//2, -n//2, n//2]
    im = ax.imshow(display, origin='lower', cmap='inferno', extent=extent)
    ax.set_xlabel('$k_x$'); ax.set_ylabel('$k_y$')
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, label='log₁₀(PSD)')

fig.tight_layout()
fig.savefig(FIG_DIR / 'step2_2d_psd.png')
plt.show()

#### 📊 How to Interpret 2D Power Spectrum

| Item | Explanation |
|------|-------------|
| **Figure meaning** | Power distribution in wavenumber domain $(k_x, k_y)$; center: $k=0$ (large scale), edges: high wavenumbers (small scale) |
| **How to read** | Bright spots/rings = energy concentrated at those wavenumbers; log scale, brighter = higher energy |
| **Good signal** | **Discrete bright spots or rings** → characteristic wavenumber exists, likely from external forcing periodicity |
| **Isotropy check** | Bright areas form **circular rings** → isotropic; distribution differs along $k_x$ or $k_y$ axes → anisotropic |
| **Key conclusion** | ① Find wavenumber $(k_x, k_y)$ with maximum energy concentration, corresponding wavelength $\lambda = L/k$ ② If obvious **isolated bright spots** (not diffuse) → likely external periodic forcing exists |

### 2.2 Radial Power Spectrum & Peak Wavenumber Detection

In [ ]:
def radial_spectrum(psd_2d):
    ny, nx = psd_2d.shape
    kx = np.fft.fftfreq(nx) * nx
    ky = np.fft.fftfreq(ny) * ny
    KX, KY = np.meshgrid(kx, ky)
    K = np.sqrt(KX**2 + KY**2)
    k_max = int(np.floor(K.max()))
    k_bins = np.arange(0, k_max + 1)
    psd_rad = np.zeros(len(k_bins))
    for i, k in enumerate(k_bins):
        mask = (K >= k - 0.5) & (K < k + 0.5)
        if mask.any():
            psd_rad[i] = psd_2d[mask].sum()
    return k_bins, psd_rad

k_bins, rad_ux = radial_spectrum(psd_ux_2d)
_, rad_uy = radial_spectrum(psd_uy_2d)
_, rad_total = radial_spectrum(psd_total_2d)

# Peak detection
rad_total_nodc = rad_total.copy()
rad_total_nodc[0] = 0
top5_idx = np.argsort(rad_total_nodc)[::-1][:5]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: three radial spectra
axes[0].semilogy(k_bins, rad_ux, 'o-', ms=3, label='$u_x$')
axes[0].semilogy(k_bins, rad_uy, 's-', ms=3, label='$u_y$')
axes[0].semilogy(k_bins, rad_total, '^-', ms=3, label='Total', color='k')
axes[0].set_xlabel('Radial wavenumber $k$')
axes[0].set_ylabel('PSD($k$)')
axes[0].set_title('Radial power spectrum')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right plot: expanded + annotated peaks
axes[1].semilogy(k_bins, rad_total, 'o-', ms=4, color='k')
for idx in top5_idx:
    axes[1].axvline(k_bins[idx], color='red', ls='--', lw=1, alpha=0.7)
    axes[1].annotate(f'k={k_bins[idx]:.0f}', (k_bins[idx], rad_total[idx]),
                     textcoords='offset points', xytext=(5, 10),
                     fontsize=9, color='red')
axes[1].set_xlabel('Radial wavenumber $k$')
axes[1].set_ylabel('PSD($k$)')
axes[1].set_title('Total radial PSD with peak wavenumbers')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'step2_radial_spectrum.png')
plt.show()

print("\n🔍 Top 5 peak wavenumbers (excluding DC):")
for idx in top5_idx:
    print(f"  k = {k_bins[idx]:.0f},  PSD = {rad_total[idx]:.4e},  "
          f"energy fraction = {rad_total[idx]/rad_total.sum()*100:.2f}%")

#### 📊 How to Interpret Radial Power Spectrum

| Item | Explanation |
|------|-------------|
| **Figure meaning** | 2D PSD integrated over radial wavenumber $k=\sqrt{k_x^2+k_y^2}$ to get 1D energy distribution |
| **How to read** | X-axis: radial wavenumber (larger = smaller spatial scales), y-axis: total energy at that scale |
| **Good signal** | **Sharp prominent peak** at some wavenumber $k_0$, far above surroundings → that scale dominates, likely from external forcing |
| **Bad signal** | Flat spectrum without obvious peaks → energy distributed uniformly across all scales |
| **Key conclusion** | ① Peak wavenumber $k_{\text{peak}}$ corresponds to characteristic wavelength $\lambda = L/k_{\text{peak}}$, where $L$ is domain size ② If peak very prominent (>20% energy), can strongly infer **external forcing injected that scale** ③ Compare $u_x$ vs $u_y$ peak positions to judge if both components dominated by same scale |

### 2.3 Temporal Frequency Power Spectrum

In [ ]:
def temporal_psd_avg(data_4d, component, dt):
    field = data_4d[..., component]
    nt = field.shape[0]
    fhat = np.fft.rfft(field, axis=0)
    psd = np.mean(np.abs(fhat)**2, axis=(1, 2)) / nt
    freqs = np.fft.rfftfreq(nt, d=dt)
    return freqs, psd

freqs_t, tpsd_ux = temporal_psd_avg(fluctuation, 0, dl.DT)
_, tpsd_uy = temporal_psd_avg(fluctuation, 1, dl.DT)
tpsd_total = tpsd_ux + tpsd_uy

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: ux vs uy
axes[0].semilogy(freqs_t[1:], tpsd_ux[1:], lw=0.8, label='$u_x$', alpha=0.8)
axes[0].semilogy(freqs_t[1:], tpsd_uy[1:], lw=0.8, label='$u_y$', alpha=0.8)
axes[0].set_xlabel('Frequency $f$')
axes[0].set_ylabel('PSD')
axes[0].set_title('Temporal PSD (spatially averaged)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right plot: total PSD + peak annotation
tpsd_total_nodc = tpsd_total.copy()
tpsd_total_nodc[0] = 0
top5_freq_idx = np.argsort(tpsd_total_nodc)[::-1][:5]

axes[1].semilogy(freqs_t[1:], tpsd_total[1:], lw=0.8, color='k')
for idx in top5_freq_idx:
    if freqs_t[idx] > 0:
        axes[1].axvline(freqs_t[idx], color='red', ls='--', lw=1, alpha=0.7)
        axes[1].annotate(f'f={freqs_t[idx]:.4f}', (freqs_t[idx], tpsd_total[idx]),
                         textcoords='offset points', xytext=(5, 10), fontsize=8, color='red')
axes[1].set_xlabel('Frequency $f$')
axes[1].set_ylabel('PSD')
axes[1].set_title('Total temporal PSD with peak frequencies')
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'step2_temporal_psd.png')
plt.show()

print("\n🔍 Top 5 peak frequencies:")
for idx in top5_freq_idx:
    if freqs_t[idx] > 0:
        print(f"  f = {freqs_t[idx]:.6f},  T = {1/freqs_t[idx]:.2f},  PSD = {tpsd_total[idx]:.4e}")

#### 📊 How to Interpret Temporal Power Spectrum

| Item | Explanation |
|------|-------------|
| **Figure meaning** | Time FFT at each grid point then spatially averaged, showing energy distribution across temporal scales |
| **How to read** | X-axis: temporal frequency $f$ (Nyquist = $1/(2\Delta t) = 2.5$), y-axis: power |
| **Good signal** | Discrete frequency peaks → system has **definite time periods**; peaks consistent with SVD mode frequencies → structural consistency |
| **Bad signal** | Broadband flat spectrum → temporal dynamics more chaotic, no characteristic time scale |
| **Key conclusion** | ① Peak frequency corresponds to period $T=1/f$, the system's characteristic time scale ② Compare $u_x$ vs $u_y$ temporal spectra: similar shape → isotropic; different → anisotropic |

---
## Step 3: Symmetry and Anisotropy Diagnosis (Project Question 4)

Diagnose system symmetry and anisotropy through:
1. **Mirror/rotation symmetry tests** of Fourier spectrum
2. **Comparison of $k_x$ vs $k_y$ axis slices**
3. **Difference in radial spectra** between $u_x$ and $u_y$ components
4. **Component energy ratio** $E_{u_x} / E_{u_y}$

### 3.1 Quantitative Symmetry Tests of Spectrum

In [ ]:
# Mirror symmetry: PSD(kx, ky) vs PSD(-kx, ky) and PSD(kx, -ky)
flip_x = np.flip(psd_total_2d, axis=1)
flip_y = np.flip(psd_total_2d, axis=0)
rotated_90 = psd_total_2d.T

norm = np.sum(psd_total_2d**2)
err_mirror_x = np.sum((psd_total_2d - flip_x)**2) / norm
err_mirror_y = np.sum((psd_total_2d - flip_y)**2) / norm
err_rot90 = np.sum((psd_total_2d - rotated_90)**2) / norm

print("=== 2D PSD Symmetry Tests ===")
print(f"x-mirror symmetry error:  {err_mirror_x:.6e}  {'✅ Symmetric' if err_mirror_x < 1e-6 else '⚠️ Asymmetric'}")
print(f"y-mirror symmetry error:  {err_mirror_y:.6e}  {'✅ Symmetric' if err_mirror_y < 1e-6 else '⚠️ Asymmetric'}")
print(f"90° rotation symmetry error: {err_rot90:.6e}  {'✅ Isotropic' if err_rot90 < 1e-6 else '⚠️ Anisotropic'}")

# Component energy ratio
E_ux = np.mean(fluctuation[..., 0]**2)
E_uy = np.mean(fluctuation[..., 1]**2)
ratio = E_ux / E_uy
print(f"\n=== Component Energy ===")
print(f"E(ux) = {E_ux:.6e}")
print(f"E(uy) = {E_uy:.6e}")
print(f"E(ux)/E(uy) = {ratio:.4f}  {'✅ Isotropic' if abs(ratio-1) < 0.05 else '⚠️ Anisotropic'}")

### 3.2 Comparison of $k_x$ vs $k_y$ Axis Slices

In [ ]:
# Take slices along kx axis (ky=0) and ky axis (kx=0)
psd_kx_slice = psd_total_2d[0, :dl.NX//2+1]   # ky=0, kx=0..32
psd_ky_slice = psd_total_2d[:dl.NY//2+1, 0]    # kx=0, ky=0..32
k_1d = np.arange(dl.NX // 2 + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: direct comparison
axes[0].semilogy(k_1d, psd_kx_slice, 'o-', ms=4, label='Along $k_x$ ($k_y=0$)')
axes[0].semilogy(k_1d, psd_ky_slice, 's-', ms=4, label='Along $k_y$ ($k_x=0$)')
axes[0].set_xlabel('Wavenumber $k$')
axes[0].set_ylabel('PSD')
axes[0].set_title('Axis slices: $k_x$ vs $k_y$')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right plot: anisotropy ratio
aniso_ratio = psd_kx_slice / (psd_ky_slice + 1e-30)
axes[1].plot(k_1d, aniso_ratio, 'o-', ms=4, color='purple')
axes[1].axhline(1.0, color='gray', ls='--', lw=1, label='Isotropic (ratio=1)')
axes[1].set_xlabel('Wavenumber $k$')
axes[1].set_ylabel('PSD($k_x$) / PSD($k_y$)')
axes[1].set_title('Anisotropy ratio')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, max(3, aniso_ratio[1:].max() * 1.2))

fig.tight_layout()
fig.savefig(FIG_DIR / 'step3_anisotropy_kx_ky.png')
plt.show()

### 3.3 Comparison of $u_x$ vs $u_y$ Component Radial Spectra

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(k_bins, rad_ux, 'o-', ms=4, label='$u_x$ radial PSD', color='steelblue')
ax.semilogy(k_bins, rad_uy, 's-', ms=4, label='$u_y$ radial PSD', color='coral')
ax.set_xlabel('Radial wavenumber $k$')
ax.set_ylabel('PSD($k$)')
ax.set_title('Component comparison: $u_x$ vs $u_y$ radial spectrum')
ax.legend()
ax.grid(True, alpha=0.3)
fig.savefig(FIG_DIR / 'step3_component_comparison.png')
plt.show()

#### 📊 How to Interpret Symmetry & Anisotropy Results

| Item | Explanation |
|------|-------------|
| **Mirror symmetry error** | Error $< 10^{-6}$ → spectrum highly symmetric in that direction; $> 10^{-2}$ → obvious asymmetry |
| **90° rotation symmetry** | Very small error → system **isotropic** ($k_x$, $k_y$ directions equivalent); large error → **anisotropic** |
| **Anisotropy ratio (ratio)** | Ratio ≈ 1 at all $k$ → isotropic; deviation from 1 at some $k$ → directional preference at that scale |
| **Component energy ratio** | $E(u_x)/E(u_y) \approx 1$ → components statistically equivalent; deviation → system has directional preference |
| **$u_x$ vs $u_y$ radial spectra** | Curves overlap → isotropic; peak positions differ → components dominated by different scales |
| **Key conclusion** | Combine above indicators to determine symmetry type: isotropic/mirror-symmetric only/fully anisotropic |

### 3.4 SVD Spatial Mode Symmetry Check

In [ ]:
half = dl.NY * dl.NX
print("=== First 6 SVD Modes Spatial Symmetry (Correlation Coefficients) ===")
print(f"{'Mode':>5} {'ux mirror-x':>12} {'ux mirror-y':>12} {'uy mirror-x':>12} {'uy mirror-y':>12}")
print("-" * 58)

for i in range(6):
    ux_m = U[:half, i].reshape(dl.NY, dl.NX)
    uy_m = U[half:, i].reshape(dl.NY, dl.NX)

    def corr(a, b):
        return np.corrcoef(a.flatten(), b.flatten())[0, 1]

    cx_ux = corr(ux_m, np.flip(ux_m, axis=1))
    cy_ux = corr(ux_m, np.flip(ux_m, axis=0))
    cx_uy = corr(uy_m, np.flip(uy_m, axis=1))
    cy_uy = corr(uy_m, np.flip(uy_m, axis=0))

    print(f"{i+1:>5} {cx_ux:>+12.3f} {cy_ux:>+12.3f} {cx_uy:>+12.3f} {cy_uy:>+12.3f}")

print()
print("Interpretation: +1 = fully symmetric, -1 = fully antisymmetric, 0 = uncorrelated")

#### 📊 How to Interpret Mode Symmetry

| Correlation Value | Meaning |
|-------------------|----------|
| ≈ **+1** | Mode **mirror-symmetric** about that axis (even symmetry) |
| ≈ **−1** | Mode **antisymmetric** about that axis (odd symmetry) |
| ≈ **0** | Mode uncorrelated with its mirror, **no symmetry** in that direction |

If multiple dominant modes exhibit consistent symmetry (e.g., all symmetric about x-axis), the **system itself possesses that directional symmetry**.

---
## 📋 Analysis Summary

### Q1: Leading Spatial Structure (SVD Analysis)
- Reaching 95% energy requires **32 modes**, 99% requires **72 modes**
- **First 4 modes dominate**: Mode 1 (35.72%) + Mode 2 (21.94%) + Mode 3 (15.65%) + Mode 4 (12.03%) = **85.34%**
- **Significant energy gap** between Mode 4→5 (12.03% drops to 0.83%), indicating first 4 modes capture core spatial structure
- Leading modes show **large-scale organized stripe patterns**, consistent with dominant wavenumber $k=1$

### Q2: Spatial Scale Distribution of Energy
- Peak wavenumber $k_{\text{peak}} = 1$ contains **84.80%** of energy, representing **domain-scale structures** (one wavelength spans full domain $L$)
- Secondary peak at $k = 4$ contains **7.64%**, a significant secondary scale
- Energy decays stepwise for $k \leq 20$, drops to noise level (~$10^{-27}$) for $k > 22$, indicating sufficient spatial resolution with minimal small-scale energy

### Q3: Inference of External Periodic Forcing
- **Presence of prominent spectral peaks**: ✅ Yes, extremely prominent
- $k = 1$ dominance at 84.80%; secondary peak at $k = 4$ (7.64%) clearly above background decay trend
- **Inferred forcing wavenumber**: $k = 4$ most likely corresponds to "prescribed spatially periodic external forcing" in project description, as it's the most prominent discrete peak above background; $k = 1$ dominance may reflect system's natural large-scale response
- **Evidence strength**: Strong — sharp concentrated peaks, highly non-uniform energy distribution

### Q4: Symmetry and Anisotropy
- **Mirror symmetry**: ❌ — x/y mirror errors of 1.92/1.95 show spectrum lacks simple mirror symmetry
- **Isotropy**: ❌ — 90° rotation error = 0.36; $E(u_x)/E(u_y) = 0.6257$ ($u_y$ ~60% higher than $u_x$), system is **clearly anisotropic**
- **SVD Mode Symmetry Features**:
  - Modes 1&2 $u_y$ components show strong x-direction mirror symmetry (correlation ≈ +0.99)
  - Modes 3&4 $u_x$ components show strong y-direction mirror symmetry (correlation ≈ +0.99)
  - System overall anisotropic, but **dominant modes have structured symmetries** (different modes symmetric in different directions)
- **Temporal Characteristics**: Temporal PSD shows broadband decay without sharp discrete peaks; energy concentrated at very low frequencies ($f \approx 0.0007 \sim 0.003$, periods $T \approx 333 \sim 1500$ time units), indicating **complex multi-scale temporal dynamics**, not simple periodic oscillation

### Additional Findings
- **Divergence**: Global divergence/velocity ratio = 0.199, system is **not strictly incompressible**, divergence contributes significantly (RMS ≈ 0.16), must be considered in physical interpretation
- **Data Statistics**: $u_x$ std = 0.707, $u_y$ std = 0.931, confirming $u_y$ component is statistically more active

> **📁 All figures saved to `figures/` directory, ready for Overleaf report inclusion.**

---
## 📊 Composite Figure for Report (Single 2×2 Figure Covering Q1–Q4)

In [ ]:
fig = plt.figure(figsize=(14, 12), constrained_layout=True)
gs = fig.add_gridspec(2, 2, hspace=0.28, wspace=0.18)

# ── (a) Singular value spectrum + cumulative energy ── Q1
ax_a = fig.add_subplot(gs[0, 0])
n_show = 50
ax_a.semilogy(np.arange(1, n_show+1), sigma[:n_show], 'o-', ms=3,
              color='steelblue', label='$\\sigma_k$')
ax_a.set_xlabel('Mode index $k$')
ax_a.set_ylabel('Singular value $\\sigma_k$')
ax_a.grid(True, alpha=0.3)

# Twin axes for cumulative energy
ax_a2 = ax_a.twinx()
ax_a2.plot(np.arange(1, n_show+1), cum_energy[:n_show]*100, 's-', ms=2.5,
           color='darkorange', label='Cumulative')
ax_a2.axhline(95, color='r', ls='--', lw=0.8)
ax_a2.set_ylabel('Cumulative energy (%)', color='darkorange')
# Annotate energy gap
ax_a.annotate('energy gap',
              xy=(5, sigma[4]), xytext=(12, sigma[2]),
              arrowprops=dict(arrowstyle='->', color='red', lw=1.2),
              fontsize=9, color='red')
ax_a.set_title('(a) SVD singular values & cumulative energy', fontsize=11)

# ── (b) Radial power spectrum + peak annotation ── Q2 & Q3
ax_b = fig.add_subplot(gs[0, 1])
ax_b.semilogy(k_bins, rad_total, 'o-', ms=4, color='k', label='Total')
# Annotate top-3 peaks (use offset to avoid overlap)
rad_nodc = rad_total.copy(); rad_nodc[0] = 0
top3 = np.argsort(rad_nodc)[::-1][:3]
offsets = [(15, 25), (15, -35), (15, 10)]  # Different offsets for each peak
for i, idx in enumerate(top3):
    frac = rad_total[idx] / rad_total.sum() * 100
    ax_b.annotate(f'k={k_bins[idx]:.0f} ({frac:.1f}%)',
                  xy=(k_bins[idx], rad_total[idx]),
                  xytext=offsets[i], textcoords='offset points',
                  arrowprops=dict(arrowstyle='->', color='red', lw=1),
                  fontsize=9, color='red', ha='left')
ax_b.set_xlabel('Radial wavenumber $k$')
ax_b.set_ylabel('PSD($k$)')
ax_b.set_title('(b) Radial power spectrum — peak wavenumbers', fontsize=11, pad=20)
ax_b.grid(True, alpha=0.3)

# ── (c) Mode 1 spatial mode ux & uy ── Q1 visual evidence
half = dl.NY * dl.NX
mode1_ux = U[:half, 0].reshape(dl.NY, dl.NX)
mode1_uy = U[half:, 0].reshape(dl.NY, dl.NX)

# Embedded subplots
gs_c = gs[1, 0].subgridspec(1, 2, wspace=0.08)
ax_c1 = fig.add_subplot(gs_c[0])
ax_c2 = fig.add_subplot(gs_c[1])

# Independent color scales for contrast
vmax_ux = np.abs(mode1_ux).max()
vmax_uy = np.abs(mode1_uy).max()
im1 = ax_c1.imshow(mode1_ux, origin='lower', cmap='RdBu_r', vmin=-vmax_ux, vmax=vmax_ux)
ax_c1.set_title(f'Mode 1 $u_x$ ({energy[0]/energy.sum()*100:.1f}%)', fontsize=10)
ax_c1.set_aspect('equal')
fig.colorbar(im1, ax=ax_c1, fraction=0.046, pad=0.04)

im2 = ax_c2.imshow(mode1_uy, origin='lower', cmap='RdBu_r', vmin=-vmax_uy, vmax=vmax_uy)
ax_c2.set_title(f'Mode 1 $u_y$ ({energy[0]/energy.sum()*100:.1f}%)', fontsize=10)
ax_c2.set_aspect('equal')
fig.colorbar(im2, ax=ax_c2, fraction=0.046, pad=0.04)

# Add overall title for (c) above subplots
fig.text(0.25, 0.35, '(c) Leading SVD spatial mode', fontsize=11,
         ha='center', va='bottom')

# ── (d) ux vs uy radial spectra + anisotropy ratio ── Q4
gs_d = gs[1, 1].subgridspec(2, 1, hspace=0.10)
ax_d1 = fig.add_subplot(gs_d[0])
ax_d2 = fig.add_subplot(gs_d[1])

ax_d1.semilogy(k_bins, rad_ux, 'o-', ms=3, label='$u_x$', color='steelblue')
ax_d1.semilogy(k_bins, rad_uy, 's-', ms=3, label='$u_y$', color='coral')
ax_d1.set_ylabel('PSD($k$)')
ax_d1.set_title('(d) Component spectra & anisotropy', fontsize=11)
ax_d1.legend(fontsize=9)
ax_d1.grid(True, alpha=0.3)

# Anisotropy ratio kx vs ky
psd_kx_sl = psd_total_2d[0, :dl.NX//2+1]
psd_ky_sl = psd_total_2d[:dl.NY//2+1, 0]
k_1d_loc = np.arange(dl.NX // 2 + 1)
ratio_line = psd_kx_sl / (psd_ky_sl + 1e-30)
ax_d2.plot(k_1d_loc, ratio_line, 'o-', ms=3, color='purple')
ax_d2.axhline(1.0, color='gray', ls='--', lw=1, label='Isotropic')
ax_d2.set_xlabel('Wavenumber $k$')
ax_d2.set_ylabel('PSD($k_x$) / PSD($k_y$)')
ax_d2.legend(fontsize=9)
ax_d2.grid(True, alpha=0.3)
ax_d2.set_ylim(0, min(5, ratio_line[1:].max() * 1.3))

fig.savefig(FIG_DIR / 'report_composite_figure.png', dpi=300)
fig.savefig(FIG_DIR / 'report_composite_figure.pdf')   # PDF vector format better for LaTeX
plt.show()
print("✅ Saved: figures/report_composite_figure.png & .pdf")